In [ ]:
from dotenv import load_dotenv
import ssl
import httpx
import truststore

load_dotenv()

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

ssl_context = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
http_client = httpx.Client(verify=ssl_context)
http_async_client = httpx.AsyncClient(verify=ssl_context)

chat_model = init_chat_model(
    model="gpt-5.4-mini",
    http_client=http_client,
    http_async_client=http_async_client
)
agent = create_agent(
    chat_model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [ ]:
from pprint import pprint

pprint(response)

In [ ]:
print(response["messages"][-1].content)